# Marshmallow — Advanced Tutorial Problems with Step-by-Step Solutions

This notebook is written in the same **tutorial rhythm** as the supplied notebook:

- introduce one idea,
- explain why it matters,
- write a small piece of code,
- run an experiment,
- ask what happens in an edge case,
- improve the schema,
- then finish with a complete solution.

The examples target **Marshmallow 4.x**.

The focus is not only on syntax. We will also practice how to design clean validation boundaries for real applications.


## Installing Marshmallow

Inside Jupyter, `%pip` installs into the Python environment associated with the current notebook kernel.

We pin Marshmallow to the 4.x major version so a future major release does not silently change the behavior of these examples.


In [ ]:
%pip install -q "marshmallow>=4.3,<5"


## Imports

We will use the same imports throughout the notebook.

Notice the major building blocks:

- `fields` describe individual values,
- `validate` provides reusable validators,
- `@pre_load` transforms data before deserialization,
- `@post_load` can build application objects,
- `@validates` validates one field,
- `@validates_schema` validates relationships between fields.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, replace
from datetime import date, datetime, timedelta, timezone
from decimal import Decimal, ROUND_HALF_UP
from enum import Enum
from pprint import pprint
from typing import Any
from uuid import UUID, uuid4

from marshmallow import (
    EXCLUDE,
    INCLUDE,
    RAISE,
    Schema,
    ValidationError,
    fields,
    post_load,
    pre_load,
    validate,
    validates,
    validates_schema,
)


## A helper for experiments

Normally, invalid input raises `ValidationError`.

That is good application behavior, but in a tutorial we often want to display an error and continue running later cells.

The helper below turns a load attempt into a small result dictionary.


In [ ]:
def inspect_load(schema: Schema, payload: Any, **kwargs):
    try:
        return {
            "ok": True,
            "value": schema.load(payload, **kwargs),
        }
    except ValidationError as exc:
        return {
            "ok": False,
            "errors": exc.messages,
            "valid_data": exc.valid_data,
        }


# Problem 1 — Build a strict inventory item

Suppose an API accepts inventory items with:

- a SKU,
- a name,
- a quantity,
- a unit price.

At first this looks like a simple schema.

But we quickly run into design questions:

- Should `"10"` be accepted as integer `10`?
- Can the name be blank?
- Can the price be negative?
- Should unknown keys be ignored?
- Should a successful load return a dictionary or a domain object?

We will answer those questions one step at a time.


## Step 1 — Define the Python object

We will deserialize into an immutable dataclass.

This gives successful input a clear application-level type.


In [ ]:
@dataclass(frozen=True)
class InventoryItem:
    sku: str
    name: str
    quantity: int
    unit_price: Decimal


## Step 2 — Start with field types only

We intentionally begin with a permissive schema.

This lets us see what field types do before we add business rules.


In [ ]:
class BasicInventorySchema(Schema):
    sku = fields.Str()
    name = fields.Str()
    quantity = fields.Int()
    unit_price = fields.Decimal(as_string=True)


In [ ]:
basic_inventory_schema = BasicInventorySchema()

raw_item = {
    "sku": "KB-001",
    "name": "Mechanical Keyboard",
    "quantity": 12,
    "unit_price": "89.99",
}

loaded_item = basic_inventory_schema.load(raw_item)

pprint(loaded_item)
print(type(loaded_item["quantity"]))
print(type(loaded_item["unit_price"]))


The unit price is now a `Decimal`.

That is useful for exact decimal values such as money.

But the schema is still permissive. Let's try several questionable inputs.


In [ ]:
experiments = [
    {},
    {"sku": "KB-001"},
    {
        "sku": "KB-001",
        "name": "",
        "quantity": "12",
        "unit_price": "-5.00",
    },
    {
        "sku": "KB-001",
        "name": "Keyboard",
        "quantity": 10,
        "unit_price": "89.99",
        "debug": True,
    },
]

for payload in experiments:
    pprint(inspect_load(basic_inventory_schema, payload))
    print("-" * 60)


## Step 3 — Add production-oriented rules

We now want:

- every field required,
- non-empty names,
- a predictable SKU,
- strict integer quantity,
- non-negative money,
- unknown fields rejected.

`strict=True` on an integer field means the string `"12"` is not silently converted to integer `12`.


In [ ]:
class InventorySchema(Schema):
    sku = fields.Str(
        required=True,
        validate=[
            validate.Length(min=3, max=24),
            validate.Regexp(r"^[A-Z0-9][A-Z0-9_-]*$"),
        ],
    )

    name = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=120),
    )

    quantity = fields.Int(
        required=True,
        strict=True,
        validate=validate.Range(min=0, max=1_000_000),
    )

    unit_price = fields.Decimal(
        required=True,
        places=2,
        rounding=ROUND_HALF_UP,
        as_string=True,
        validate=validate.Range(min=Decimal("0.00")),
    )

    class Meta:
        unknown = RAISE

    @post_load
    def make_item(self, data, **kwargs):
        return InventoryItem(**data)


## Step 4 — Test the complete solution

Notice that the result is now an `InventoryItem`, not a dictionary.


In [ ]:
inventory_schema = InventorySchema()

item = inventory_schema.load({
    "sku": "KB-001",
    "name": "Mechanical Keyboard",
    "quantity": 12,
    "unit_price": "89.995",
})

print(item)
print(type(item))
pprint(inventory_schema.dump(item))


The input had three decimal places, but the schema quantized the value to two.

Now let's verify the strict rules.


In [ ]:
invalid_inventory = [
    {
        "sku": "bad sku",
        "name": "Keyboard",
        "quantity": 2,
        "unit_price": "10.00",
    },
    {
        "sku": "GOOD-1",
        "name": "",
        "quantity": 2,
        "unit_price": "10.00",
    },
    {
        "sku": "GOOD-1",
        "name": "Keyboard",
        "quantity": "2",
        "unit_price": "10.00",
    },
    {
        "sku": "GOOD-1",
        "name": "Keyboard",
        "quantity": 2,
        "unit_price": "-0.01",
    },
]

for payload in invalid_inventory:
    pprint(inspect_load(inventory_schema, payload))
    print("-" * 60)


### What we learned

A robust schema describes much more than types.

It can also define:

- coercion policy,
- required values,
- range rules,
- unknown-field behavior,
- conversion into domain objects.


# Problem 2 — Normalize before validating

Suppose a signup form sends names and email addresses with accidental whitespace.

It may also send a two-letter country code in lowercase.

If normalization is part of the input contract, we can keep it inside the schema using `@pre_load`.


## Step 1 — Define the result object


In [ ]:
@dataclass(frozen=True)
class Subscriber:
    first_name: str
    last_name: str
    email: str
    country_code: str


## Step 2 — See why normalization matters

A name made only of spaces can pass a simple length check because spaces still count as characters.

Likewise, a valid-looking email surrounded by spaces may fail email validation.


In [ ]:
class UnnormalizedSubscriberSchema(Schema):
    first_name = fields.Str(required=True, validate=validate.Length(min=1))
    last_name = fields.Str(required=True, validate=validate.Length(min=1))
    email = fields.Email(required=True)
    country_code = fields.Str(
        required=True,
        validate=validate.Regexp(r"^[A-Z]{2}$"),
    )

messy_subscriber = {
    "first_name": "   Ada   ",
    "last_name": "  Lovelace ",
    "email": " ADA@EXAMPLE.COM ",
    "country_code": "gb",
}

pprint(inspect_load(
    UnnormalizedSubscriberSchema(),
    messy_subscriber,
))


## Step 3 — Normalize in `@pre_load`

A good pattern is:

1. copy the input dictionary,
2. normalize strings,
3. return the normalized dictionary,
4. let field validation run afterward.

Copying avoids mutating a dictionary still owned by the caller.


In [ ]:
class SubscriberSchema(Schema):
    first_name = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=80),
    )
    last_name = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=80),
    )
    email = fields.Email(required=True)
    country_code = fields.Str(
        required=True,
        validate=validate.Regexp(r"^[A-Z]{2}$"),
    )

    class Meta:
        unknown = RAISE

    @pre_load
    def normalize(self, data, **kwargs):
        data = dict(data)

        for key in ("first_name", "last_name", "email", "country_code"):
            value = data.get(key)
            if isinstance(value, str):
                data[key] = value.strip()

        if isinstance(data.get("email"), str):
            data["email"] = data["email"].lower()

        if isinstance(data.get("country_code"), str):
            data["country_code"] = data["country_code"].upper()

        return data

    @post_load
    def make_subscriber(self, data, **kwargs):
        return Subscriber(**data)


In [ ]:
subscriber_schema = SubscriberSchema()

subscriber = subscriber_schema.load(messy_subscriber)

print(subscriber)


Now consider a first name containing only spaces.

The spaces are removed first, so the validator sees an empty string and can reject it correctly.


In [ ]:
pprint(inspect_load(
    subscriber_schema,
    {
        "first_name": "   ",
        "last_name": "Lovelace",
        "email": "ada@example.com",
        "country_code": "GB",
    },
))


### Design principle

Normalization and validation are different jobs.

A clean sequence is usually:

1. normalize alternate representations,
2. validate the normalized result.


# Problem 3 — Separate readable and writable fields

Imagine an account API.

The client should submit:

- `displayName`,
- `email`,
- `password`.

The server should return:

- `id`,
- `displayName`,
- `email`,
- `createdAt`.

The password must never be serialized back to the client.

This gives us three useful field options:

- `load_only`,
- `dump_only`,
- `data_key`.


## Step 1 — Define the API schema

The Python field name stays `display_name`.

The external JSON key becomes `displayName`.


In [ ]:
class AccountSchema(Schema):
    account_id = fields.UUID(
        dump_only=True,
        data_key="id",
    )

    display_name = fields.Str(
        required=True,
        data_key="displayName",
        validate=validate.Length(min=2, max=80),
    )

    email = fields.Email(required=True)

    password = fields.Str(
        required=True,
        load_only=True,
        validate=validate.Length(min=12, max=256),
    )

    created_at = fields.AwareDateTime(
        dump_only=True,
        data_key="createdAt",
    )

    class Meta:
        unknown = RAISE


## Step 2 — Load a registration request

The loaded dictionary uses Python-friendly names.


In [ ]:
account_schema = AccountSchema()

registration = account_schema.load({
    "displayName": "Barbara Liskov",
    "email": "barbara@example.com",
    "password": "a-long-example-password",
})

pprint(registration)


## Step 3 — Dump a response

The object below contains a `password` key only to prove that `load_only=True` prevents it from appearing in output.


In [ ]:
server_account = {
    "account_id": uuid4(),
    "display_name": "Barbara Liskov",
    "email": "barbara@example.com",
    "password": "THIS MUST NOT LEAK",
    "created_at": datetime.now(timezone.utc),
}

response = account_schema.dump(server_account)

pprint(response)

assert "password" not in response


## Step 4 — What if a client supplies a server-owned field?

Because `id` is dump-only, it is not a normal input field.

Combined with strict unknown-field handling, the request is rejected.


In [ ]:
pprint(inspect_load(
    account_schema,
    {
        "id": str(uuid4()),
        "displayName": "Barbara Liskov",
        "email": "barbara@example.com",
        "password": "a-long-example-password",
    },
))


# Problem 4 — Nested schemas with recipes

A recipe contains ingredients.

Each ingredient has its own validation rules.

Instead of duplicating ingredient validation inside the recipe, we create a reusable nested schema.


## Step 1 — Define the Python objects


In [ ]:
@dataclass(frozen=True)
class Ingredient:
    name: str
    amount: Decimal
    unit: str


@dataclass(frozen=True)
class Recipe:
    title: str
    servings: int
    ingredients: tuple[Ingredient, ...]


## Step 2 — Solve the smaller schema first

Before nesting anything, make sure one ingredient can be validated correctly.


In [ ]:
class IngredientSchema(Schema):
    name = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=100),
    )

    amount = fields.Decimal(
        required=True,
        as_string=True,
        places=2,
        rounding=ROUND_HALF_UP,
        validate=validate.Range(min=Decimal("0.01")),
    )

    unit = fields.Str(
        required=True,
        validate=validate.OneOf(
            ["g", "kg", "ml", "l", "tsp", "tbsp", "piece"]
        ),
    )

    class Meta:
        unknown = RAISE

    @post_load
    def make_ingredient(self, data, **kwargs):
        return Ingredient(**data)


In [ ]:
ingredient = IngredientSchema().load({
    "name": "Flour",
    "amount": "250",
    "unit": "g",
})

print(ingredient)


## Step 3 — Nest ingredients inside the recipe

A recipe has a list of nested ingredients.


In [ ]:
class RecipeSchema(Schema):
    title = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=150),
    )

    servings = fields.Int(
        required=True,
        strict=True,
        validate=validate.Range(min=1, max=100),
    )

    ingredients = fields.List(
        fields.Nested(IngredientSchema),
        required=True,
        validate=validate.Length(min=1, max=100),
    )

    class Meta:
        unknown = RAISE

    @post_load
    def make_recipe(self, data, **kwargs):
        data["ingredients"] = tuple(data["ingredients"])
        return Recipe(**data)


In [ ]:
recipe_payload = {
    "title": "Simple Pancakes",
    "servings": 4,
    "ingredients": [
        {"name": "Flour", "amount": "250", "unit": "g"},
        {"name": "Milk", "amount": "300", "unit": "ml"},
        {"name": "Egg", "amount": "2", "unit": "piece"},
    ],
}

recipe = RecipeSchema().load(recipe_payload)

print(recipe)
print(type(recipe.ingredients[0]))


## Step 4 — Inspect nested error paths

The second ingredient below has several problems.

Marshmallow reports the list index of the bad nested element.


In [ ]:
bad_recipe_payload = {
    "title": "Broken Recipe",
    "servings": 2,
    "ingredients": [
        {"name": "Flour", "amount": "100", "unit": "g"},
        {"name": "", "amount": "-5", "unit": "bucket"},
    ],
}

pprint(inspect_load(
    RecipeSchema(),
    bad_recipe_payload,
))


# Problem 5 — Cross-field validation with bank transfers

Sometimes every individual field is valid, but the combination is not.

A bank transfer may contain:

- a valid source account,
- a valid destination account,
- a valid amount,

and still be invalid if the source and destination are the same.

That rule belongs at schema level.


## Step 1 — Start with field-level validation


In [ ]:
class TransferSchema(Schema):
    source_account = fields.Str(
        required=True,
        data_key="sourceAccount",
        validate=validate.Regexp(r"^[A-Z]{2}[0-9A-Z]{10,32}$"),
    )

    destination_account = fields.Str(
        required=True,
        data_key="destinationAccount",
        validate=validate.Regexp(r"^[A-Z]{2}[0-9A-Z]{10,32}$"),
    )

    amount = fields.Decimal(
        required=True,
        as_string=True,
        places=2,
        validate=validate.Range(
            min=Decimal("0.01"),
            max=Decimal("1000000.00"),
        ),
    )

    currency = fields.Str(
        required=True,
        validate=validate.OneOf(["EUR", "USD", "GBP"]),
    )

    class Meta:
        unknown = RAISE


All individual fields in the next transfer are valid.

But the transaction itself is nonsensical.


In [ ]:
self_transfer = {
    "sourceAccount": "BG80BANK000000000001",
    "destinationAccount": "BG80BANK000000000001",
    "amount": "100.00",
    "currency": "EUR",
}

pprint(TransferSchema().load(self_transfer))


## Step 2 — Add a schema-level invariant

`@validates_schema` receives the already-deserialized values.

That makes it ideal for relationships between several fields.


In [ ]:
class StrictTransferSchema(TransferSchema):
    @validates_schema
    def validate_relationships(self, data, **kwargs):
        if data["source_account"] == data["destination_account"]:
            raise ValidationError({
                "destination_account": [
                    "Destination account must differ from source account."
                ]
            })


In [ ]:
pprint(inspect_load(
    StrictTransferSchema(),
    self_transfer,
))


## Step 3 — Add another rule involving amount and currency

Suppose GBP transfers cannot exceed 50,000.

This rule depends on two fields.


In [ ]:
class FinalTransferSchema(TransferSchema):
    @validates_schema
    def validate_relationships(self, data, **kwargs):
        errors = {}

        if data["source_account"] == data["destination_account"]:
            errors.setdefault("destination_account", []).append(
                "Destination account must differ from source account."
            )

        if (
            data["currency"] == "GBP"
            and data["amount"] > Decimal("50000.00")
        ):
            errors.setdefault("amount", []).append(
                "GBP transfers cannot exceed 50000.00."
            )

        if errors:
            raise ValidationError(errors)


In [ ]:
pprint(inspect_load(
    FinalTransferSchema(),
    {
        "sourceAccount": "GB11BANK000000000001",
        "destinationAccount": "GB11BANK000000000002",
        "amount": "75000.00",
        "currency": "GBP",
    },
))


# Problem 6 — Enum fields for support tickets

When the set of allowed values belongs to the domain, a Python `Enum` can be clearer than scattering string literals across the codebase.

We will model ticket priority and ticket status.


In [ ]:
class TicketPriority(Enum):
    LOW = "low"
    NORMAL = "normal"
    HIGH = "high"
    URGENT = "urgent"


class TicketStatus(Enum):
    OPEN = "open"
    IN_PROGRESS = "in_progress"
    RESOLVED = "resolved"
    CLOSED = "closed"


## Step 1 — Deserialize strings into enum members

With `by_value=True`, external values such as `"urgent"` map to `TicketPriority.URGENT`.


In [ ]:
class TicketSchema(Schema):
    title = fields.Str(
        required=True,
        validate=validate.Length(min=5, max=200),
    )

    priority = fields.Enum(
        TicketPriority,
        required=True,
        by_value=True,
    )

    status = fields.Enum(
        TicketStatus,
        required=True,
        by_value=True,
    )

    class Meta:
        unknown = RAISE


In [ ]:
ticket = TicketSchema().load({
    "title": "Database connection keeps dropping",
    "priority": "urgent",
    "status": "open",
})

pprint(ticket)
print(ticket["priority"] is TicketPriority.URGENT)


## Step 2 — Add a state-dependent rule

For this exercise, suppose a closed ticket cannot retain urgent priority.

The rule depends on two enum fields, so we validate it at schema level.


In [ ]:
class BusinessTicketSchema(TicketSchema):
    @validates_schema
    def validate_state(self, data, **kwargs):
        if (
            data["status"] is TicketStatus.CLOSED
            and data["priority"] is TicketPriority.URGENT
        ):
            raise ValidationError(
                "A closed ticket cannot retain urgent priority."
            )


In [ ]:
pprint(inspect_load(
    BusinessTicketSchema(),
    {
        "title": "Database connection keeps dropping",
        "priority": "urgent",
        "status": "closed",
    },
))


# Problem 7 — PATCH-style partial loading

Create and update operations have different semantics.

On creation, fields can be required.

On PATCH, the client often sends only the fields it wants to change.

Marshmallow supports this with `partial=True`.


## Step 1 — A normal create schema


In [ ]:
class ProfileSchema(Schema):
    display_name = fields.Str(
        required=True,
        validate=validate.Length(min=2, max=80),
    )

    email = fields.Email(required=True)

    bio = fields.Str(
        required=True,
        validate=validate.Length(max=500),
    )

    class Meta:
        unknown = RAISE


Loading one field normally fails because the other required fields are missing.


In [ ]:
pprint(inspect_load(
    ProfileSchema(),
    {"bio": "A newly updated biography."},
))


## Step 2 — Use `partial=True`

Missing required fields are tolerated, but supplied values are still validated.


In [ ]:
profile_schema = ProfileSchema()

patch = profile_schema.load(
    {"bio": "A newly updated biography."},
    partial=True,
)

pprint(patch)


An invalid email remains invalid even in a partial update.


In [ ]:
pprint(inspect_load(
    profile_schema,
    {"email": "not-an-email"},
    partial=True,
))


## Step 3 — Apply the validated patch

Marshmallow validates the patch.

The application decides how to apply it to the domain object.


In [ ]:
@dataclass(frozen=True)
class UserProfile:
    display_name: str
    email: str
    bio: str


original_profile = UserProfile(
    display_name="Linus",
    email="linus@example.com",
    bio="Original biography",
)

patch_data = profile_schema.load(
    {"bio": "Updated biography"},
    partial=True,
)

updated_profile = replace(
    original_profile,
    **patch_data,
)

print("original:", original_profile)
print("updated: ", updated_profile)


### Important caution

If a schema has a `@post_load` hook that always constructs a complete object, `partial=True` may not be enough.

The object constructor may still require every field.

In larger applications, separate schemas for create commands, patch commands, and responses are often clearer.


# Problem 8 — Unknown-field policies

Marshmallow can:

- reject unknown fields,
- remove them,
- keep them.

These correspond to:

- `RAISE`,
- `EXCLUDE`,
- `INCLUDE`.

Let's see the exact behavior on the same payload.


In [ ]:
class TelemetrySchema(Schema):
    device_id = fields.UUID(
        required=True,
        data_key="deviceId",
    )
    temperature = fields.Float(required=True)


In [ ]:
telemetry_payload = {
    "deviceId": str(uuid4()),
    "temperature": 22.4,
    "firmwareVersion": "7.1.0",
}

for policy in (RAISE, EXCLUDE, INCLUDE):
    print("Policy:", policy)
    pprint(inspect_load(
        TelemetrySchema(unknown=policy),
        telemetry_payload,
    ))
    print("-" * 60)


### Choosing a policy

`RAISE` is a strong default for write APIs and commands because it catches typos and unexpected client behavior.

`EXCLUDE` can be useful when consuming third-party payloads that evolve independently and you intentionally ignore fields you do not model.

`INCLUDE` is useful in pass-through or generic ingestion systems, but it should be a deliberate choice.


# Problem 9 — Computed output with `fields.Method`

Suppose an event session has a start time and an end time.

The API response should also include `durationMinutes`.

The client should not submit that value because it can be calculated from the trusted timestamps.

A dump-only `Method` field is a good fit.


In [ ]:
@dataclass(frozen=True)
class Session:
    title: str
    starts_at: datetime
    ends_at: datetime


In [ ]:
class SessionSchema(Schema):
    title = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=150),
    )

    starts_at = fields.AwareDateTime(
        required=True,
        data_key="startsAt",
    )

    ends_at = fields.AwareDateTime(
        required=True,
        data_key="endsAt",
    )

    duration_minutes = fields.Method(
        serialize="get_duration_minutes",
        dump_only=True,
        data_key="durationMinutes",
    )

    class Meta:
        unknown = RAISE

    @validates_schema
    def validate_times(self, data, **kwargs):
        if data["ends_at"] <= data["starts_at"]:
            raise ValidationError({
                "ends_at": [
                    "Session must end after it starts."
                ]
            })

    def get_duration_minutes(self, obj):
        seconds = (
            obj.ends_at - obj.starts_at
        ).total_seconds()

        return int(seconds // 60)

    @post_load
    def make_session(self, data, **kwargs):
        return Session(**data)


## Step 2 — Load, then dump

The computed field is added only during serialization.


In [ ]:
session_schema = SessionSchema()

session = session_schema.load({
    "title": "Advanced Python",
    "startsAt": "2026-09-01T10:00:00+03:00",
    "endsAt": "2026-09-01T11:45:00+03:00",
})

print(session)
pprint(session_schema.dump(session))


Now try an impossible time range.


In [ ]:
pprint(inspect_load(
    session_schema,
    {
        "title": "Impossible Session",
        "startsAt": "2026-09-01T12:00:00+03:00",
        "endsAt": "2026-09-01T11:00:00+03:00",
    },
))


# Problem 10 — Write a custom percentage field

Sometimes a domain value deserves its own reusable field.

We want the external representation:

```text
12.5%
```

to become the internal value:

```python
Decimal("0.125")
```

The field must also reject values below 0% or above 100%.


## Step 1 — Define the conversion rules

A custom field implements:

- `_deserialize` for input,
- `_serialize` for output.


In [ ]:
class PercentageField(fields.Field):
    default_error_messages = {
        "invalid": "Not a valid percentage.",
        "range": "Percentage must be between 0% and 100%.",
    }

    def _deserialize(self, value, attr, data, **kwargs):
        if not isinstance(value, str):
            raise ValidationError(
                self.error_messages["invalid"]
            )

        if not value.endswith("%"):
            raise ValidationError(
                self.error_messages["invalid"]
            )

        try:
            number = Decimal(value[:-1].strip())
        except Exception as exc:
            raise ValidationError(
                self.error_messages["invalid"]
            ) from exc

        if number < 0 or number > 100:
            raise ValidationError(
                self.error_messages["range"]
            )

        return number / Decimal("100")

    def _serialize(self, value, attr, obj, **kwargs):
        if value is None:
            return None

        percentage = (
            Decimal(value) * Decimal("100")
        ).normalize()

        return f"{percentage}%"


## Step 2 — Reuse the field in a discount schema


In [ ]:
class DiscountSchema(Schema):
    code = fields.Str(required=True)
    rate = PercentageField(required=True)

    class Meta:
        unknown = RAISE


In [ ]:
discount_schema = DiscountSchema()

discount = discount_schema.load({
    "code": "SPRING",
    "rate": "12.5%",
})

pprint(discount)
print(type(discount["rate"]))
pprint(discount_schema.dump(discount))


## Step 3 — Test invalid percentage representations


In [ ]:
for rate in [
    "12.5",
    "-1%",
    "101%",
    "hello%",
]:
    print("Input:", rate)
    pprint(inspect_load(
        discount_schema,
        {
            "code": "TEST",
            "rate": rate,
        },
    ))
    print("-" * 50)


# Problem 11 — Recursive schemas for threaded comments

A comment can contain replies.

Each reply is another comment.

This is a recursive data structure.

Marshmallow can represent it by nesting the schema inside itself.


In [ ]:
@dataclass(frozen=True)
class Comment:
    author: str
    text: str
    replies: tuple["Comment", ...] = field(
        default_factory=tuple
    )


## Step 1 — Use a callable nested schema

The lambda delays construction of the nested schema.


In [ ]:
class CommentSchema(Schema):
    author = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=80),
    )

    text = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=2000),
    )

    replies = fields.List(
        fields.Nested(lambda: CommentSchema()),
        load_default=list,
    )

    class Meta:
        unknown = RAISE

    @post_load
    def make_comment(self, data, **kwargs):
        data["replies"] = tuple(data["replies"])
        return Comment(**data)


## Step 2 — Load multiple levels of nesting


In [ ]:
comment_payload = {
    "author": "Alice",
    "text": "Has anyone used recursive schemas?",
    "replies": [
        {
            "author": "Bob",
            "text": "Yes, they work well for tree data.",
            "replies": [
                {
                    "author": "Carol",
                    "text": "The resulting Python objects are convenient.",
                }
            ],
        }
    ],
}

comment = CommentSchema().load(comment_payload)

print(comment)


## Step 3 — Traverse the domain objects

This confirms that nested dictionaries became nested `Comment` objects.


In [ ]:
def print_comment_tree(comment: Comment, depth=0):
    print(
        "  " * depth
        + f"{comment.author}: {comment.text}"
    )

    for reply in comment.replies:
        print_comment_tree(reply, depth + 1)


print_comment_tree(comment)


## Step 4 — Serialize the tree back to native data


In [ ]:
pprint(CommentSchema().dump(comment))


# Problem 12 — Bulk validation with `many=True`

Imports and batch APIs often validate a list of records.

With `many=True`, Marshmallow reports errors using list indexes.

This makes it possible to tell a user exactly which rows are invalid.


In [ ]:
class EmployeeImportSchema(Schema):
    employee_id = fields.Int(
        required=True,
        strict=True,
        data_key="employeeId",
        validate=validate.Range(min=1),
    )

    name = fields.Str(
        required=True,
        validate=validate.Length(min=2, max=100),
    )

    email = fields.Email(required=True)

    department = fields.Str(
        required=True,
        validate=validate.OneOf([
            "engineering",
            "finance",
            "sales",
            "operations",
        ]),
    )

    class Meta:
        unknown = RAISE
        index_errors = True


## Step 1 — Prepare a batch with different failures


In [ ]:
employee_rows = [
    {
        "employeeId": 1,
        "name": "Alice",
        "email": "alice@example.com",
        "department": "engineering",
    },
    {
        "employeeId": 2,
        "name": "B",
        "email": "bad-email",
        "department": "engineering",
    },
    {
        "employeeId": "3",
        "name": "Charlie",
        "email": "charlie@example.com",
        "department": "legal",
    },
    {
        "employeeId": 4,
        "name": "Dana",
        "department": "finance",
    },
]


## Step 2 — Validate the whole collection

Look at the shape of `exc.messages`.

The outer keys identify the indexes of invalid records.


In [ ]:
employee_schema = EmployeeImportSchema(many=True)

try:
    employee_schema.load(employee_rows)
except ValidationError as exc:
    print("Errors:")
    pprint(exc.messages)

    print("\nValid data preserved by Marshmallow:")
    pprint(exc.valid_data)


### Application-level decision

`valid_data` does not automatically mean "commit these records".

Your import process still needs a policy:

- all-or-nothing,
- or partial acceptance.

That is a transaction/business decision, not merely a schema decision.


# Problem 13 — Build a reusable camelCase base schema

Many APIs use:

- camelCase externally,
- snake_case internally.

Adding `data_key` to every field can become repetitive.

Marshmallow's `on_bind_field` hook lets a base schema apply a naming convention automatically.


## Step 1 — Convert snake_case to camelCase


In [ ]:
def snake_to_camel(name: str) -> str:
    first, *rest = name.split("_")

    return first + "".join(
        part.title()
        for part in rest
    )


In [ ]:
for name in [
    "first_name",
    "created_at",
    "shipping_address",
    "id",
]:
    print(name, "->", snake_to_camel(name))


## Step 2 — Apply the convention when fields are bound

If a field already has an explicit `data_key`, we leave it alone.

Otherwise we derive the external key automatically.


In [ ]:
class CamelCaseSchema(Schema):
    class Meta:
        unknown = RAISE

    def on_bind_field(self, field_name, field_obj):
        if field_obj.data_key is None:
            field_obj.data_key = snake_to_camel(
                field_name
            )


## Step 3 — Create a normal child schema

The child schema contains only Pythonic field names.


In [ ]:
class AuditEventSchema(CamelCaseSchema):
    event_id = fields.UUID(required=True)
    event_type = fields.Str(required=True)
    created_at = fields.AwareDateTime(required=True)
    actor_email = fields.Email(required=True)


In [ ]:
audit_payload = {
    "eventId": str(uuid4()),
    "eventType": "user.login",
    "createdAt": "2026-08-07T15:00:00+00:00",
    "actorEmail": "alice@example.com",
}

audit_event = AuditEventSchema().load(
    audit_payload
)

pprint(audit_event)
pprint(AuditEventSchema().dump(audit_event))


### Why this is an advanced pattern

The naming policy is now hidden in a base class.

That reduces repetition, but it also makes behavior less obvious to readers.

Use this pattern when the convention is stable, project-wide, and well documented.


# Problem 14 — Output projections with `only` and `exclude`

Sometimes one object has several legitimate representations.

A customer list endpoint may need only:

- id,
- display name.

A detailed endpoint may return more.

Marshmallow can create field projections without defining a new schema class for every small variation.


In [ ]:
class CustomerRecordSchema(Schema):
    customer_id = fields.UUID(
        data_key="id"
    )
    display_name = fields.Str(
        data_key="displayName"
    )
    email = fields.Email()
    phone = fields.Str()
    created_at = fields.AwareDateTime(
        data_key="createdAt"
    )


In [ ]:
customer_record = {
    "customer_id": uuid4(),
    "display_name": "Ada Lovelace",
    "email": "ada@example.com",
    "phone": "+44-000-000-000",
    "created_at": datetime.now(timezone.utc),
}


## Step 1 — Full representation


In [ ]:
pprint(
    CustomerRecordSchema().dump(
        customer_record
    )
)


## Step 2 — Compact representation with `only`


In [ ]:
compact_schema = CustomerRecordSchema(
    only=(
        "customer_id",
        "display_name",
    )
)

pprint(
    compact_schema.dump(
        customer_record
    )
)


## Step 3 — Remove a field with `exclude`


In [ ]:
public_schema = CustomerRecordSchema(
    exclude=("phone",)
)

pprint(
    public_schema.dump(
        customer_record
    )
)


### Security reminder

`only` and `exclude` shape output.

They do not replace authorization.

Your application must still decide whether a caller is allowed to access the underlying resource.


# Problem 15 — Capstone: conference registration

We will now combine many ideas from the notebook.

A conference registration contains:

- an attendee,
- a ticket type,
- one or more session selections,
- a registration timestamp,
- an optional discount.

The rules will involve:

- nested schemas,
- enums,
- timezone-aware datetimes,
- a custom field,
- list constraints,
- schema-level invariants,
- computed output,
- dataclass construction,
- automatic camelCase conversion.


## Step 1 — Define the domain types


In [ ]:
class TicketType(Enum):
    STANDARD = "standard"
    STUDENT = "student"
    VIP = "vip"


@dataclass(frozen=True)
class Attendee:
    name: str
    email: str


@dataclass(frozen=True)
class SessionSelection:
    session_id: UUID
    title: str


@dataclass(frozen=True)
class ConferenceRegistration:
    registration_id: UUID
    attendee: Attendee
    ticket_type: TicketType
    sessions: tuple[SessionSelection, ...]
    registered_at: datetime
    discount_rate: Decimal | None = None


## Step 2 — Build the attendee schema first

We solve nested pieces independently before assembling the full schema.


In [ ]:
class AttendeeSchema(CamelCaseSchema):
    name = fields.Str(
        required=True,
        validate=validate.Length(min=2, max=100),
    )

    email = fields.Email(required=True)

    @pre_load
    def normalize_attendee(self, data, **kwargs):
        data = dict(data)

        if isinstance(data.get("name"), str):
            data["name"] = data["name"].strip()

        if isinstance(data.get("email"), str):
            data["email"] = (
                data["email"]
                .strip()
                .lower()
            )

        return data

    @post_load
    def make_attendee(self, data, **kwargs):
        return Attendee(**data)


In [ ]:
attendee = AttendeeSchema().load({
    "name": "  Grace Hopper ",
    "email": " GRACE@EXAMPLE.COM ",
})

print(attendee)


## Step 3 — Build the session-selection schema


In [ ]:
class SessionSelectionSchema(CamelCaseSchema):
    session_id = fields.UUID(required=True)

    title = fields.Str(
        required=True,
        validate=validate.Length(min=1, max=150),
    )

    @post_load
    def make_selection(self, data, **kwargs):
        return SessionSelection(**data)


## Step 4 — Assemble the registration schema

The server generates `registration_id`, so it is dump-only.

`discount_rate` reuses the custom percentage field.

`session_count` is computed during serialization.


In [ ]:
class ConferenceRegistrationSchema(CamelCaseSchema):
    registration_id = fields.UUID(
        dump_only=True
    )

    attendee = fields.Nested(
        AttendeeSchema,
        required=True,
    )

    ticket_type = fields.Enum(
        TicketType,
        required=True,
        by_value=True,
    )

    sessions = fields.List(
        fields.Nested(
            SessionSelectionSchema
        ),
        required=True,
        validate=validate.Length(
            min=1,
            max=8,
        ),
    )

    registered_at = fields.AwareDateTime(
        required=True
    )

    discount_rate = PercentageField(
        allow_none=True,
        load_default=None,
    )

    session_count = fields.Method(
        serialize="get_session_count",
        dump_only=True,
    )

    @validates_schema
    def validate_registration(
        self,
        data,
        **kwargs,
    ):
        errors = {}

        sessions = data.get(
            "sessions",
            [],
        )

        session_ids = [
            session.session_id
            for session in sessions
        ]

        if len(session_ids) != len(set(session_ids)):
            errors.setdefault(
                "sessions",
                [],
            ).append(
                "The same session cannot be selected twice."
            )

        if (
            data.get("ticket_type")
            is TicketType.VIP
            and data.get("discount_rate") is not None
            and data["discount_rate"] > Decimal("0.25")
        ):
            errors.setdefault(
                "discount_rate",
                [],
            ).append(
                "VIP registrations cannot receive more than a 25% discount."
            )

        if errors:
            raise ValidationError(errors)

    def get_session_count(self, obj):
        return len(obj.sessions)

    @post_load
    def make_registration(self, data, **kwargs):
        data["sessions"] = tuple(
            data["sessions"]
        )

        return ConferenceRegistration(
            registration_id=uuid4(),
            **data,
        )


## Step 5 — Load a valid registration

Before running the code, predict:

- What Python type will `ticket_type` become?
- What type will each session become?
- What will `"10%"` become internally?
- Where will the registration ID come from?


In [ ]:
registration_payload = {
    "attendee": {
        "name": "  Grace Hopper ",
        "email": " GRACE@EXAMPLE.COM ",
    },
    "ticketType": "standard",
    "sessions": [
        {
            "sessionId": str(uuid4()),
            "title": "Compilers in Practice",
        },
        {
            "sessionId": str(uuid4()),
            "title": "Distributed Systems",
        },
    ],
    "registeredAt": "2026-08-07T15:30:00+00:00",
    "discountRate": "10%",
}

registration_schema = ConferenceRegistrationSchema()

registration = registration_schema.load(
    registration_payload
)

print(registration)
print(type(registration.ticket_type))
print(type(registration.sessions[0]))
print(registration.discount_rate)


## Step 6 — Serialize the domain object

The base schema converts field names to camelCase automatically.

The output also contains the computed `sessionCount`.


In [ ]:
pprint(
    registration_schema.dump(
        registration
    )
)


## Step 7 — Duplicate session selections should fail

We deliberately reuse the same UUID twice.


In [ ]:
duplicate_id = str(uuid4())

duplicate_payload = {
    "attendee": {
        "name": "Grace Hopper",
        "email": "grace@example.com",
    },
    "ticketType": "standard",
    "sessions": [
        {
            "sessionId": duplicate_id,
            "title": "Session A",
        },
        {
            "sessionId": duplicate_id,
            "title": "Session A again",
        },
    ],
    "registeredAt": "2026-08-07T15:30:00+00:00",
}

pprint(inspect_load(
    registration_schema,
    duplicate_payload,
))


## Step 8 — Test the VIP discount rule


In [ ]:
vip_discount_payload = {
    "attendee": {
        "name": "Grace Hopper",
        "email": "grace@example.com",
    },
    "ticketType": "vip",
    "sessions": [
        {
            "sessionId": str(uuid4()),
            "title": "VIP Architecture Workshop",
        }
    ],
    "registeredAt": "2026-08-07T15:30:00+00:00",
    "discountRate": "40%",
}

pprint(inspect_load(
    registration_schema,
    vip_discount_payload,
))


## Step 9 — Trigger several independent errors

A useful API should provide structured error information.

The following payload intentionally breaks multiple rules at once.


In [ ]:
bad_registration_payload = {
    "attendee": {
        "name": " ",
        "email": "not-an-email",
    },
    "ticketType": "super-premium",
    "sessions": [],
    "registeredAt": "not-a-datetime",
    "discountRate": "200%",
    "unexpectedField": True,
}

pprint(inspect_load(
    registration_schema,
    bad_registration_payload,
))


# Final review

Across the notebook, Marshmallow performed several different jobs.

## Deserialization

External values became richer Python values:

- strings became `Decimal`,
- strings became `UUID`,
- strings became timezone-aware `datetime`,
- strings became enum members,
- dictionaries became dataclass instances.

## Validation

We validated:

- formats,
- lengths,
- numeric ranges,
- enumerated values,
- list sizes,
- nested values,
- relationships between several fields.

## Normalization

`@pre_load` converted messy but acceptable input into a canonical representation before validation.

## Serialization

Application objects became native, JSON-friendly representations with:

- external field names,
- read-only fields,
- omitted write-only fields,
- computed values.


# Extra advanced exercises

These are intentionally left as practice after the fully solved problems above.

## Exercise A — Nested partial updates

Create an `OrganizationSchema` with a nested `AddressSchema`.

Load this PATCH payload:

```python
{
    "address": {
        "city": "Sofia"
    }
}
```

Use partial loading so the other address fields are not required.

## Exercise B — Promotion time windows

Create a `PromotionSchema` with:

- `starts_at`,
- `ends_at`,
- `discount`.

Rules:

- both datetimes must be timezone-aware,
- `ends_at` must be after `starts_at`,
- the promotion may last at most 90 days,
- discount must be between 0% and 80%.

## Exercise C — Custom identifier field

Accept values such as:

```text
USR_01HZX3Q8A4
ORD_01HZX3Q8A4
```

Validate the prefix and convert the value into a small domain object.

## Exercise D — Bulk conference registration

Use:

```python
ConferenceRegistrationSchema(many=True)
```

Validate several registrations and inspect indexed errors.

## Exercise E — Split command and response schemas

Create:

- `CreateRegistrationSchema`,
- `RegistrationResponseSchema`.

Compare that approach with one schema containing `load_only` and `dump_only` fields.


# Common mistakes

## Mistake 1 — Using the old Marshmallow 2 result API

Old tutorials may show:

```python
result = schema.load(data)
result.data
result.errors
```

Modern Marshmallow does not work that way.

`load()` returns the deserialized result directly and raises `ValidationError` for invalid input.

## Mistake 2 — Treating `dump()` as input validation

`dump()` is primarily serialization.

Validate untrusted input with `load()` or `validate()`.

## Mistake 3 — Mutating the caller's dictionary in `@pre_load`

Prefer making a copy before normalization:

```python
data = dict(data)
```

## Mistake 4 — Using float for exact monetary values

Prefer `Decimal` where exact decimal behavior matters.

## Mistake 5 — Accepting unknown fields accidentally

Choose `RAISE`, `EXCLUDE`, or `INCLUDE` deliberately.

## Mistake 6 — Putting every rule into `@validates_schema`

Keep single-field rules close to their fields.

Reserve schema-level validation for relationships between fields.

## Mistake 7 — Confusing validation with authorization

Schemas validate data shape and values.

They do not replace permission checks.


# Reference links

Official Marshmallow documentation:

- https://marshmallow.readthedocs.io/
- https://marshmallow.readthedocs.io/en/stable/quickstart.html
- https://marshmallow.readthedocs.io/en/stable/nesting.html
- https://marshmallow.readthedocs.io/en/stable/custom_fields.html
- https://marshmallow.readthedocs.io/en/stable/extending/pre_and_post_processing_methods.html
- https://marshmallow.readthedocs.io/en/stable/extending/schema_validation.html

A good way to study this notebook is to modify one value at a time and observe how the error structure changes.
